> **ADVERTENCIA — CALIBRACIÓN HISTÓRICA / LEGACY**
>
> Este notebook utilizó la conversión histórica SCCM → g/L/h: **1 SCCM ≈ 0.05890515 g L^-1 h^-1** para 2 L, basada en **Vm = 22.414 L/mol** y **sin aplicar K_CO2 = 0.74**. Se conserva deliberadamente sin cambios para reproducibilidad y no representa la conversión física actualmente adoptada.
>
> El experimento corregido utiliza **1 SCCM ≈ 0.04043919 g L^-1 h^-1** y documenta la comparación en `co2_sccm_correction_experiment_2026.ipynb`.
>
> **ESTE NOTEBOOK NO DEBE SER MODIFICADO PARA USAR LA NUEVA CONVERSIÓN.** Se mantiene como registro histórico y no debe volver a ejecutarse.

# CO₂ 2026: cero de sensor, inicio gradual y validación cruzada

## Resultado que se evalúa

Esta corrida corrige dos fuentes de la cola inicial artificial: la activación casi escalonada de la fuente biológica y la liberación gaseosa nula hasta superar la saturación. La comparación primaria conserva la respuesta de N pero usa la liberación por umbral anterior; el modelo lento legado se conserva como referencia histórica.

- Antes de filtrar, cada corrida recibe una corrección de cero en sccm estimada con el percentil 10 de sus primeras 12 h. El cero no se comparte entre campañas.
- Los archivos exponen canales físicos F1–F3. DOE-F06 conserva la etiqueta solicitada `sensor 6`, aunque se adquiere por F3; el offset se estima por corrida y no depende de agrupar ambas etiquetas.
- La química define el intervalo admisible de activación. Dos parámetros compartidos por matriz sitúan el comienzo y la duración de una rampa suave dentro de esa evidencia; no se estima un retardo libre por lote.
- `O2_qmax_mg_gdw_h` y `O2_initial_scale` se estiman por matriz junto con transferencia, solubilidad y ganancia.
- La fase gaseosa puede recibir un flujo pequeño desde que existe CO₂ disuelto; ya no se exige superar (C^*_{CO_2}) para abandonar exactamente cero.
- El efecto del pulso se construye como la diferencia causal entre simulaciones upstream con y sin adición de N. `pulse_t_rise_h` distribuye su disponibilidad en el tiempo y `pulse_activity_gain` representa capacidad metabólica adicional sin convertirla automáticamente en biomasa.
- En LAB el pulso se ubica por cruce de densidad 1040 g/L; en DOE-F0X se respeta el tiempo de proceso ya registrado en cada fermentación.
- Todos los gráficos de perfil de CO₂ marcan el instante del pulso nutricional.
- A setpoint ≤15.5 °C se usa un umbral operacional más conservador de 0.10 g CO₂ L⁻¹ h⁻¹; en el resto, 0.05.
- Se calibran por separado mosto sintético y natural. DOE-F06 y LAB012 son holdouts completos; después se cruzan los modelos sin reajuste.
- Se excluyen LAB001–LAB003, LAB009 y DOE-F02 por perfiles no confiables.

La analítica química sigue definiendo t=0 y el término del proceso. Esta es una validación de la capa CO₂ condicionada por los drivers upstream disponibles, no una recalibración end-to-end del modelo de fermentación.

## Estructura del modelo

Para la corrida (r), primero se corrige el cero instrumental en la señal nativa:

\[
b_r=\max\left[0,Q_{0.10}\{y_{sccm}(t):t\leq t_0+12\,h\}\right],\qquad
y_{corr}(t)=\max[y_{sccm}(t)-b_r,0]
\]

La conversión a g L⁻¹ h⁻¹, el filtrado de transientes y el suavizado ocurren después. Esta regla supone que el extremo inferior de las primeras 12 h representa cero instrumental; si ya existe flujo biológico durante toda esa ventana, puede sustraer parte de la señal real.

La primera muestra química que cumple pérdida de glucosa+fructosa ≥5 g/L o aumento de etanol ≥2 g/L marca el extremo superior del intervalo de activación. La muestra química anterior fija el extremo inferior. La fuente usa una rampa *smoothstep* continua:

\[
t_s=t_L+f_s(t_U-t_L),\quad
\Delta t=f_d(t_U-t_L),\quad
a_{chem}=3z^2-2z^3,\quad
z=\operatorname{clip}\!\left(\frac{t-t_s}{\Delta t},0,1\right)
\]

Los parámetros (f_s) y (f_d) son compartidos por matriz. Esto permite un aumento pequeño y gradual desde (t_s), pero mantiene el comienzo ligado al bracket químico en vez de introducir una latencia independiente por fermentación.

\[
\frac{dO_2}{dt}=-q_{O_2,max}X\frac{O_2}{K_{O_2}+O_2},\qquad
\phi_{ana}=\frac{K_{ana}^{h}}{K_{ana}^{h}+O_2^{h}}
\]

\[
q_{prod}=a_{chem}(t)\left[q_{bio}\left(f_{Crabtree}+(1-f_{Crabtree})\phi_{ana}\right)+q_{resp}\right]
\]

Para una adición en \(t_N\), la fracción utilizada es causal:

\[
f_N(t)=\operatorname{clip}\left(\frac{t-t_N}{t_{rise}},0,1\right),\qquad
q_{bio,N}=\left[1+(g_N-1)f_N(t)\right]q_{bio,0}
+f_N(t)\left(q_{bio,+N}-q_{bio,0}\right)
\]

La biomasa usa la misma diferencia con/sin pulso, pero sin \(g_N\). Así, el término de actividad puede capturar el aumento de capacidad de transporte descrito por el grupo de Sablayrolles sin imponer crecimiento ficticio.

\[
C^*_{CO_2}=s_{sat}\,1.69\,e^{-0.032(T-20)}e^{0.0016E}e^{-0.0012(G+F)}
\]

\[
\frac{dC_{CO_2}}{dt}=q_{prod}-q_{gas},\qquad
q_{gas}=\min\!\left[k_{release}C_{CO_2}
\left(f_0+(1-f_0)\frac{C_{CO_2}}{C_{CO_2}+C^*_{CO_2}}\right),
\frac{C_{CO_2}}{\Delta t}+q_{prod}\right],\quad f_0=0.05
\]

La formulación anterior, usada como comparador directo, tenía (q_{gas}=k_{release}\operatorname{softplus}(C_{CO_2}-C^*_{CO_2})): eso mantenía la predicción pegada a cero hasta acumular suficiente CO₂ disuelto. El nuevo piso (f_0) representa liberación sub-saturada pequeña y evita ese umbral duro.

El inicio observado se define como el primero de tres puntos horarios consecutivos sobre el mayor valor entre: límite de detección local y línea base inicial +10 % del rango dinámico. Además se reporta la primera emisión sostenida sobre 0.005 g L⁻¹ h⁻¹ y la duración visual 2–10 % del ascenso. El ajuste conserva el residuo de tiempo de inicio y, en lotes pulsados, el desfase del máximo durante las 72 h posteriores a la adición, además de los residuos del perfil completo.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython import get_ipython
from IPython.display import display

get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT.name != "pyomo-doe" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if ROOT.name != "pyomo-doe":
    raise RuntimeError("Execute this notebook from the repository or a descendant")

FM = ROOT / "fermentation_model"
for path in (FM, FM / "laboratory_2026", FM / "pilot_2025"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from laboratory_2026 import run_co2_matrix_cross_validation_2026 as analysis

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
print("Repository:", ROOT)
print("Current model:", analysis.MODEL_NAME)
print("Direct comparator:", analysis.THRESHOLD_RELEASE_MODEL_NAME)
print("Historical comparator:", analysis.LEGACY_MODEL_NAME)
print("Holdouts:", analysis.HOLDOUTS)
print("Excluded:", analysis.EXCLUDED_BATCHES)

## Ejecución completa y partición experimental

In [ ]:
result = analysis.run_analysis(n_starts=5, max_nfev=300, seed=20260812)

display(result["exclusions"])
display(result["inventory"][[
    "matrix", "experiment_code", "batch", "lot", "calibratable", "split",
    "acquisition_channel", "sensor_id", "sensor_zero_offset_sccm",
    "n_co2_hourly", "chemistry_first_h", "chemistry_last_h", "co2_first_h",
    "co2_last_h", "co2_peak_g_l_h", "median_setpoint_c",
    "n_artifacts_replaced", "n_left_censored_hourly", "note"
]])
display(result["natural_nutrient_pulses"][[
    "batch", "calendar_t_h", "density_crossing_t_h", "density_bracket_width_h",
    "model_pulse_time_h", "timing_source", "amount_N_kg_m3",
    "calendar_product", "calendar_dose", "excluded_from_co2_analysis"
]])
fig = analysis.plot_process_timeline_alignment(
    result["inventory"], result["nutrient_pulses"], save=False
)
plt.show()

La separación se hace por fermentación completa. Los pulsos LAB aportan 0.14 kg N m⁻³; el pulso inicial se ignora porque el YAN inicial ya está en t=0. En todos los LAB con cruce disponible, el tiempo del pulso se obtiene por interpolación del cruce de densidad 1040 g/L. Un bracket químico >24 h se conserva, pero se etiqueta como incertidumbre de tiempo. En DOE-F0X se usa el tiempo de proceso existente, sin recalcularlo por densidad ni calendario.

## Filtrado y sensibilidad del sensor

In [ ]:
display(result["qc_summary"][[
    "matrix", "experiment_code", "batch", "acquisition_channel", "sensor_id",
    "sensor_zero_offset_sccm", "sensor_zero_offset_g_l_h", "median_setpoint_c",
    "n_artifacts_replaced", "n_sampling_window_transients",
    "n_other_short_transients", "negative_signed_fraction",
    "n_left_censored_hourly", "n_cold_low_sensitivity_hourly",
    "smoothing_roughness_ratio", "smoothing_integral_ratio", "smoothing_peak_ratio"
]])
display(result["sensor_zero_offsets"])
fig = analysis.plot_sensor_zero_correction(
    result["sensor_qc"], result["sensor_zero_offsets"], save=False
)
plt.show()
fig = analysis.plot_data_overview(
    result["observations"], result["nutrient_pulses"], save=False
)
plt.show()
fig = analysis.plot_sensor_filter_examples(
    result["sensor_qc"], result["sampling_schedule"],
    result["nutrient_pulses"], save=False
)
plt.show()

El primer gráfico comprueba la corrección de cero antes de cualquier reconstrucción. El offset se estima por corrida porque los canales muestran deriva entre campañas. Luego, las excursiones asociadas a muestreo se reconstruyen sólo cuando el evento completo está respaldado por niveles pre/post consistentes; se aplica mediana robusta de 3 h y Savitzky–Golay cuadrático de 5 h. Los puntos bajo el umbral local aportan una penalización unilateral si el modelo excede el límite.

## Temperatura y evidencia química de activación

In [ ]:
display(result["driver_diagnostics"][[
    "matrix", "batch", "temperature_used_min_c", "temperature_used_mean_c",
    "temperature_used_max_c", "chemical_activity_lower_h",
    "chemical_activity_upper_h", "chemical_activity_center_h",
    "initial_o2_saturation_base_mg_l", "base_qprod_peak_g_l_h",
    "csat_base_min_g_l", "csat_base_max_g_l"
]])
display(result["temperature_alignment"])
fig = analysis.plot_temperature_profiles(
    result["observations"], result["temperature_inputs"],
    result["nutrient_pulses"], save=False
)
plt.show()

La temperatura medida/reconstruida entra tanto a los drivers cinéticos como a la solubilidad de CO₂; el setpoint queda como referencia. La tabla permite verificar, lote por lote, el intervalo químico que condiciona el encendido de la fuente.

## Calibración por matriz

In [ ]:
display(result["fit_parameters"])
best_starts = (
    result["fit_starts"].sort_values("wsse_equal_batch")
    .groupby("matrix", as_index=False).first()
)
display(best_starts[[
    "matrix", "success", "nfev", "wsse_equal_batch", "kCO2_release_h",
    "CO2sat_scale", "O2_qmax_mg_gdw_h", "O2_initial_scale",
    "pulse_t_rise_h", "pulse_activity_gain",
    "chem_activation_start_fraction", "chem_activation_duration_fraction",
    "matrix_gain"
]])
fig = analysis.plot_parameter_comparison(result["fit_parameters"], save=False)
plt.show()
for matrix in ("synthetic", "natural"):
    fig = analysis.plot_calibration_overlays(result["predictions"], matrix, save=False)
    plt.show()

`active_bound=True` indica que el dato sólo acota el parámetro en el borde permitido. Debe leerse como tensión estructural o falta de identificabilidad, no como una estimación interior resuelta. `chem_activation_start_fraction` ubica el inicio dentro del bracket químico y `chem_activation_duration_fraction` escala la duración por el ancho de ese bracket. `pulse_activity_gain=1` significa ausencia de modulación sobre la actividad preexistente; >1 es boost y <1 es atenuación.

## Estimabilidad de inicio y duración

In [ ]:
display(result["jacobian_identifiability"])
display(
    result["local_parameter_correlations"].query(
        "parameter_1 in ['chem_activation_start_fraction', 'chem_activation_duration_fraction']"
    )[["matrix", "parameter_1", "parameter_2", "local_log_parameter_correlation"]]
    .sort_values(["matrix", "parameter_1", "local_log_parameter_correlation"])
)
display(result["loo_activation_summary"])
display(result["loo_activation_estimates"][[
    "matrix", "omitted_batch", "success", "nfev",
    "chem_activation_start_fraction", "chem_activation_duration_fraction"
]])
display(result["activation_profiles"])
fig = analysis.plot_activation_identifiability(
    result["activation_profiles"], result["loo_activation_summary"], save=False
)
plt.show()

El Jacobiano evalúa sensibilidad local en log-parámetros; un número de condición alto indica direcciones compensables. Los perfiles fijan uno de los dos parámetros y reoptimizan todos los demás. La franja naranjo muestra cuánto cambia la estimación al retirar una fermentación completa. La línea horizontal de 5 % es sólo un umbral descriptivo de sensibilidad del objetivo, no un intervalo de confianza de verosimilitud.

## ¿Se corrigió la cola inicial?

In [ ]:
native_transition = result["source_transition_diagnostics"].query("native_matrix_fit").copy()
display(native_transition[[
    "target_matrix", "experiment_code", "batch", "median_setpoint_c",
    "cold_operation", "chemical_activity_lower_h", "chemical_activity_upper_h",
    "source_10pct_peak_onset_h", "o2_below_anaerobic_halfpoint_h",
    "observed_onset_h", "predicted_onset_h", "onset_delay_h",
    "O2_qmax_mg_gdw_h", "O2_initial_scale", "n_pulse_time_h",
    "pulse_t_rise_h", "pulse_activity_gain",
    "chem_activation_start_fraction", "chem_activation_duration_fraction",
    "pulse_utilization_complete_h",
    "observed_postpulse_peak_h", "predicted_postpulse_peak_h",
    "postpulse_peak_delay_h"
]])
fig = analysis.plot_initial_release_comparison(
    result["predictions"], result["previous_predictions"],
    result["batch_metrics"], result["previous_batch_metrics"], save=False
)
plt.show()
fig = analysis.plot_onset_model_comparison(
    result["batch_metrics"], result["previous_batch_metrics"], save=False
)
plt.show()
fig = analysis.plot_pulse_response_comparison(
    result["predictions"], result["previous_predictions"],
    result["batch_metrics"], result["previous_batch_metrics"],
    result["source_transition_diagnostics"], save=False
)
plt.show()

El primer gráfico es el control visual principal: amplía el inicio de cada lote y compara liberación por umbral (naranjo) con liberación continua (azul). La línea horizontal marca 0.005 g L⁻¹ h⁻¹; el texto cuantifica la primera emisión y la duración 2–10 %. El segundo resume el error de inicio. El tercero amplía cada lote pulsado: línea magenta = adición; línea morada = término de la disponibilidad gradual estimada.

## Validación retenida y transferencia cruzada

In [ ]:
validation_columns = [
    "scenario", "calibration_matrix", "target_matrix", "experiment_code", "batch",
    "n", "n_total", "n_left_censored", "rmse_g_l_h", "nrmse_peak", "bias_g_l_h",
    "correlation", "r2", "integral_ratio_pred_over_observed_lower_bound",
    "observed_onset_h", "predicted_onset_h", "onset_delay_h",
    "predicted_first_emission_0p005_h", "observed_visual_rise_duration_h",
    "predicted_visual_rise_duration_h", "visual_rise_duration_error_h",
    "pulse_time_h", "observed_postpulse_peak_h", "predicted_postpulse_peak_h",
    "postpulse_peak_delay_h"
]
display(result["validation"][validation_columns])
display(result["model_comparison"])
display(result["filter_impact"])
fig = analysis.plot_validation(result["predictions"], result["validation"], save=False)
plt.show()
fig = analysis.plot_validation_model_comparison(
    result["predictions"], result["previous_predictions"],
    result["validation"], result["previous_validation"], save=False
)
plt.show()

`model_comparison` usa exactamente los mismos holdouts para la liberación por umbral anterior y la liberación continua nueva, conservando la misma respuesta finita al pulso. Valores negativos en las columnas `*_change_continuous_minus_threshold_release` significan mejora. `filter_impact` compara los ajustes crudo y filtrado sobre el mismo soporte cuantificable.

## Diagnóstico por fermentación

In [ ]:
display(result["batch_metrics"][[
    "calibration_matrix", "target_matrix", "experiment_code", "batch", "role",
    "n", "n_left_censored", "rmse_g_l_h", "nrmse_peak", "bias_g_l_h",
    "correlation", "integral_ratio_pred_over_observed_lower_bound",
    "observed_onset_h", "predicted_onset_h", "onset_delay_h", "pulse_time_h",
    "predicted_first_emission_0p005_h", "observed_visual_rise_duration_h",
    "predicted_visual_rise_duration_h", "visual_rise_duration_error_h",
    "observed_postpulse_peak_h", "predicted_postpulse_peak_h",
    "postpulse_peak_delay_h"
]].sort_values(["calibration_matrix", "target_matrix", "batch"]))

## Experimento pendiente para distinguir sensor de biología

El umbral dependiente de temperatura sigue siendo operacional, no un LOD/LOQ certificado. Para identificarlo se requiere un ensayo a 15, 18 y 21 °C con el mismo medio y volumen, referencia independiente de CO₂ y escalones de 0.01–0.30 g CO₂ L⁻¹ h⁻¹. En cada temperatura deben estimarse LOD, LOQ, sesgo, repetibilidad y tiempo de respuesta.

La corrección por percentil inicial también es operacional. Debe reemplazarse por blancos de gas sin fermentación medidos al comienzo y al final de cada corrida en cada canal. Eso separaría offset, deriva y flujo biológico temprano sin depender de una ventana temporal elegida.

La validación biológica complementaria debe iniciar con medición química frecuente durante las primeras 48–72 h, especialmente en frío, para estrechar el intervalo de activación. Sin ese muestreo, la química sólo identifica un intervalo y no un instante exacto.

## Resumen reproducible

In [ ]:
for row in result["validation"].itertuples(index=False):
    direction = f"{row.calibration_matrix} → {row.target_matrix}"
    print(
        f"{direction:22s} | {row.experiment_code:7s} | "
        f"RMSE={row.rmse_g_l_h:.3f} | r={row.correlation:.2f} | "
        f"onset error={row.onset_delay_h:.1f} h | censored={row.n_left_censored}"
    )

active = result["fit_parameters"].query("active_bound")
if not active.empty:
    print("\nAdvertencia: parámetros en límite")
    display(active)

comparison = result["model_comparison"]
print("\nDiagnóstico local de estimabilidad:")
display(result["jacobian_identifiability"])
print("\nEstabilidad leave-one-batch-out de inicio/duración:")
display(result["loo_activation_summary"])
print("\nCambio medio de |error de inicio|, nuevo - anterior [h]:",
      comparison["abs_onset_error_change_continuous_minus_threshold_release"].mean())
print("Cambio medio de RMSE, nuevo - anterior [g/L/h]:",
      comparison["rmse_change_continuous_minus_threshold_release"].mean())
print("\nCambio medio de |error de duración visual 2-10%|, nuevo - anterior [h]:",
      comparison["abs_visual_rise_duration_error_change_continuous_minus_threshold_release"].mean())
print("\nCambio medio de |error del máximo post-pulso|, nuevo - anterior [h]:",
      comparison["abs_postpulse_peak_error_change_continuous_minus_threshold_release"].mean())
print("\nArtefactos guardados en:", analysis.RESULTS_DIR.relative_to(ROOT))

## Artefactos

La ejecución guarda offsets por sensor/corrida, señal antes y después de corregir cero, observaciones filtradas, soporte químico, pulsos, temperatura, parámetros y multistart, perfiles de objetivo, Jacobiano, leave-one-batch-out, predicciones, métricas, validaciones, figuras y manifiesto JSON. El notebook ejecutado conserva todas las tablas y gráficos visibles.